In [1]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_25439/2998990347.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
loader = TextLoader("../data/speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='Virat Kohli\n\nVirat Kohli is an Indian international cricketer and one of the greatest batsmen in the history of cricket. He was born on November 5, 1988, in Delhi, India.\n\nKohli made his international debut for India in 2008 in a One Day International (ODI) match against Sri Lanka. He became the captain of the Indian cricket team across all formats and is known for his aggressive leadership and exceptional consistency.\n\nVirat Kohli is a right-handed batsman. He has scored more than 80 international centuries across Test, ODI, and T20I cricket, making him one of the highest century scorers in the history of the sport.\n\nHe played a crucial role in India\'s victory in the 2011 ICC Cricket World Cup and was also part of the team that won the 2013 ICC Champions Trophy.\n\nKohli has represented Royal Challengers Bengaluru (RCB) in the Indian Premier League (IPL) since the league began in 2008. He is the highest run scorer in 

In [3]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

In [6]:
embedding=OllamaEmbeddings(model="nomic-embed-text")  
vectordb=Chroma.from_documents(documents=splits,embedding=embedding)
vectordb

In [ ]:
## query it
query = "Kohli is batter or bowler"
docs = vectordb.similarity_search(query)
docs[0].page_content

'Playing Style:\nVirat Kohli is famous for chasing targets successfully in ODI cricket. Because of his remarkable performances while chasing, he is often called the "Chase Master." His strengths include cover drives, quick running between the wickets, and the ability to build innings under pressure.'

In [ ]:
## Saving to the disk
vectordb=Chroma.from_documents(documents=splits,embedding=embedding,persist_directory="./chroma_db")


In [ ]:
# load from disk
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
docs=db2.similarity_search(query)
print(docs[0].page_content)

To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.


In [ ]:
## similarity Search With Score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.', metadata={'source': 'speech.txt'}),
  15573.843876438417),
 (Document(page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.', metadata={'source': 'speech.txt'}),
  18241.307592374058),
 

In [ ]:
### Retriever option
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'